In [1]:
# Install library
!pip install emoji Sastrawi nltk pandas
!pip install python-Levenshtein
!pip install --upgrade git+https://github.com/ariaghora/mpstemmer.git
!pip install imbalanced-learn scikit-learn
!pip install tqdm

# Import library umum
import re
import string
import requests
import csv
from io import StringIO
from datetime import datetime as dt

import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

# Text preprocessing
import emoji
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from mpstemmer import MPStemmer

# Machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE

# Deep learning libraries (Keras / TensorFlow)
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Conv1D, GlobalMaxPooling1D,
    LSTM, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# NLTK downloads
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')  # jika diperlukan untuk tokenisasi khusus


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.7 MB/s eta 0:00:00
  Cloning https://github.com/ariaghora/mpstemmer.git to /tmp/pip-req-build-hi0go4jy
  Running command git clone --filter=blob:none --quiet https://github.com/ariaghora/mpstemmer.git /tmp/pip-req-build-hi0go4jy
  Resolved https://github.com/ariaghora/mpstemmer.git to commit 25a5fd923af163a7eac3a5ec976984156ca8fa8b
  Preparing metadata (setup.py) ... done
  Created wheel for mpstemmer: filename=mpstemmer-0.1.0-py3-none-any.whl size=99799 sha256=3f6e9a3a754953622f4d117ea6eff4ac10facb2f6a517243ccf4e35194905e87
  Stored in directory: /tmp/pip-ephem-wheel-cache-61qukd14/wheels/85/1f/78/fccc1e7e90613593e0a66304a3711d678e3013034d91534182
Successfully built mpstemmer


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
# Load data
df_raw = pd.read_csv('data_raw.csv')

print("Ukuran dataset:", df_raw.shape)
df_raw.head()

Ukuran dataset: (15000, 11)


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,d4ed2636-3cef-4eb5-8b3e-7dc1efd3ac46,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aplikasi Dompet Digital yang cukup bagus ada t...,5,699,1.60.0,2025-10-09 21:46:08,NaN,NaN,1.60.0
1,e7effc5f-85b4-4934-9a57-7761b149981f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"sebelumnya saya pake aplikasi lain, setelah ta...",5,130,1.60.0,2025-10-18 23:23:59,NaN,NaN,1.60.0
2,2e9f4216-d3c0-4a24-8cce-684d3646f37d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,saat ingin transfer uang ke aplikasi bank kok ...,5,166,1.60.0,2025-10-18 06:17:09,NaN,NaN,1.60.0
3,6637bb64-62b6-4a2c-a6ea-40bb067d32d8,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,koin² yang saya kumpulkan beberapa hari terakh...,1,148,1.60.0,2025-10-11 04:13:43,"Hai Kak Jensen, mohon maaf atas kendalanya. Pa...",2025-10-11 10:10:58,1.60.0
4,bd06e982-1f75-4ebd-acbf-5628846e9c2b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"aplikasi ini bagus untuk transaksi,tetapi seri...",3,75,1.60.0,2025-10-07 10:48:25,"Mohon maaf atas kendalanya, Kak Budi. Terkait ...",2025-10-07 14:26:11,1.60.0


In [3]:
#cek missing values
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              15000 non-null  object
 1   userName              15000 non-null  object
 2   userImage             15000 non-null  object
 3   content               15000 non-null  object
 4   score                 15000 non-null  int64 
 5   thumbsUpCount         15000 non-null  int64 
 6   reviewCreatedVersion  13477 non-null  object
 7   at                    15000 non-null  object
 8   replyContent          7876 non-null   object
 9   repliedAt             7876 non-null   object
 10  appVersion            13477 non-null  object
dtypes: int64(2), object(9)
memory usage: 1.3+ MB


In [4]:
# Memilih kolom penting
df_selected = df_raw[['content']]
df_selected.duplicated().sum()


np.int64(19)

In [5]:
# Menghapus data duplikat
df_clean = df_selected.drop_duplicates()
df_clean.duplicated().sum()

np.int64(0)

In [6]:
df_clean

,content
0,aplikasi Dompet Digital yang cukup bagus ada t...
1,"sebelumnya saya pake aplikasi lain, setelah ta..."
2,saat ingin transfer uang ke aplikasi bank kok ...
3,koin² yang saya kumpulkan beberapa hari terakh...
4,"aplikasi ini bagus untuk transaksi,tetapi seri..."
...,...
14995,Disaat mau gunakan transfer selalu keluar tuli...
14996,GoPay adalah sebuah terobosan dalam dunia keua...
14997,Saya beri nilai 1 karna apk ini tidak bisa di ...
14998,Untuk saran ke developernya mungkin untuk pros...


In [7]:

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka
    text = re.sub(r'(.)\1{2,}', r'\1', text)           # normalisasi huruf berulang

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text
# Fungsi Case Folding (mengubah huruf menjadi lower case)
def casefoldingText(text):
    return text.lower()

# Fungsi Mengganti Kata Slang
def load_slangwords(local_file_path):
    slang_dict = {}

    # slang lokal
    with open(local_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if ':' in line:
                slang, formal = line.strip().split(':', 1)
                slang_dict[slang.lower()] = formal.lower()

    # slang dari GitHub
    url = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
    df_slang = pd.read_csv(url)
    slang_github = dict(zip(df_slang['slang'], df_slang['formal']))
    slang_dict.update({k.lower(): v.lower() for k, v in slang_github.items()})

    return slang_dict

def fix_slangwords(text, slang_dict):
    words = text.split()
    fixed_words = [slang_dict.get(word.lower(), word) for word in words]
    return ' '.join(fixed_words)

# Fungsi Tokenizing
def tokenizingText(text):
    return text.split()

def load_combined_stopwords(local_path_gopay=None):
    stoplist = set(stopwords.words('indonesian'))
    stoplist.update(stopwords.words('english'))

    if local_path_gopay:
        try:
            with open(local_path_gopay, 'r', encoding='utf-8') as f:
                local_stop = [line.strip() for line in f if line.strip()]
            stoplist.update(local_stop)
        except FileNotFoundError:
            print("⚠️ File stopwords_gopay.txt tidak ditemukan, lanjut pakai bawaan NLTK saja.")

    return stoplist

stemmer = MPStemmer()

def filtering_and_stemming(words, combined_stopwords):
    domain_keep = {
        # Negasi
        'tidak', 'bukan', 'nggak', 'ga', 'gak', 'tak', 'ndak',
        # Penyangkalan ringan
        'belum', 'tanpa',
        # Penegasan/pembanding
        'kurang', 'lebih', 'sangat', 'terlalu', 'sekali', 'amat',
        # Penguatan opini
        'banget', 'parah', 'lumayan',
        # Konteks sentimen
        'hanya', 'cuma', 'malah', 'justru'
    }

    filtered = [word for word in words if word not in combined_stopwords or word in domain_keep]

    # Stemming pakai MPStemmer
    stemmed = [stemmer.stem(word) for word in filtered]

    return stemmed


def toSentence(list_words):
    return ' '.join(word for word in list_words)

combined_stopwords_url = "https://raw.githubusercontent.com/louisowen6/NLP_bahasa_resources/master/combined_stop_words.txt"
slang_path = "slang_gopay.txt"

# load resource
slang_dict = load_slangwords(slang_path)
combined_stopwords = load_combined_stopwords('stopwords_gopay.txt')

# pipeline
df_clean['text_clean']     = df_clean['content'].apply(cleaningText)
df_clean['text_casefold']  = df_clean['text_clean'].apply(casefoldingText)
df_clean['text_slang']     = df_clean['text_casefold'].apply(lambda x: fix_slangwords(x, slang_dict))
df_clean['text_token']     = df_clean['text_slang'].apply(tokenizingText)
df_clean['text_filtered']  = df_clean['text_token'].apply(lambda x: filtering_and_stemming(x, combined_stopwords))
df_clean['text_ready']     = df_clean['text_filtered'].apply(toSentence)

df_clean[['content', 'text_ready']].head(5)

/tmp/ipython-input-2494751043.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['text_clean']     = df_clean['content'].apply(cleaningText)
/tmp/ipython-input-2494751043.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['text_casefold']  = df_clean['text_clean'].apply(casefoldingText)
/tmp/ipython-input-2494751043.py:97: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats 

,content,text_ready
0,aplikasi Dompet Digital yang cukup bagus ada t...,bagus gratis murah loh bintang pokok bug level...
1,"sebelumnya saya pake aplikasi lain, setelah ta...",tidak pindah mantap pokok cuma sayang sekali m...
2,saat ingin transfer uang ke aplikasi bank kok ...,bilang jaring putus pikir habis coba buka lanc...
3,koin² yang saya kumpulkan beberapa hari terakh...,koin² kumpul pet tidak akumulasi checkin misi ...
4,"aplikasi ini bagus untuk transaksi,tetapi seri...",bagus transaksitetapi sekali erormasalah jarin...


In [8]:
df_clean

,content,text_clean,text_casefold,text_slang,text_token,text_filtered,text_ready
0,aplikasi Dompet Digital yang cukup bagus ada t...,aplikasi Dompet Digital yang cukup bagus ada t...,aplikasi dompet digital yang cukup bagus ada t...,aplikasi dompet digital yang cukup bagus ada t...,"[aplikasi, dompet, digital, yang, cukup, bagus...","[bagus, gratis, murah, loh, bintang, pokok, bu...",bagus gratis murah loh bintang pokok bug level...
1,"sebelumnya saya pake aplikasi lain, setelah ta...",sebelumnya saya pake aplikasi lain setelah tau...,sebelumnya saya pake aplikasi lain setelah tau...,sebelumnya saya pakai aplikasi lain setelah ta...,"[sebelumnya, saya, pakai, aplikasi, lain, sete...","[tidak, pindah, mantap, pokok, cuma, sayang, s...",tidak pindah mantap pokok cuma sayang sekali m...
2,saat ingin transfer uang ke aplikasi bank kok ...,saat ingin transfer uang ke aplikasi bank kok ...,saat ingin transfer uang ke aplikasi bank kok ...,saat ingin transfer uang ke aplikasi bank kok ...,"[saat, ingin, transfer, uang, ke, aplikasi, ba...","[bilang, jaring, putus, pikir, habis, coba, bu...",bilang jaring putus pikir habis coba buka lanc...
3,koin² yang saya kumpulkan beberapa hari terakh...,koin² yang saya kumpulkan beberapa hari terakh...,koin² yang saya kumpulkan beberapa hari terakh...,koin² yang saya kumpulkan beberapa hari terakh...,"[koin², yang, saya, kumpulkan, beberapa, hari,...","[koin², kumpul, pet, tidak, akumulasi, checkin...",koin² kumpul pet tidak akumulasi checkin misi ...
4,"aplikasi ini bagus untuk transaksi,tetapi seri...",aplikasi ini bagus untuk transaksitetapi serin...,aplikasi ini bagus untuk transaksitetapi serin...,aplikasi ini bagus untuk transaksitetapi serin...,"[aplikasi, ini, bagus, untuk, transaksitetapi,...","[bagus, transaksitetapi, sekali, erormasalah, ...",bagus transaksitetapi sekali erormasalah jarin...
...,...,...,...,...,...,...,...
14995,Disaat mau gunakan transfer selalu keluar tuli...,Disaat mau gunakan transfer selalu keluar tuli...,disaat mau gunakan transfer selalu keluar tuli...,disaat mau gunakan transfer selalu keluar tuli...,"[disaat, mau, gunakan, transfer, selalu, kelua...","[saat, tulis, tidak, jaring, bayar, scan, gaga...",saat tulis tidak jaring bayar scan gagal baik ...
14996,GoPay adalah sebuah terobosan dalam dunia keua...,GoPay adalah sebuah terobosan dalam dunia keua...,gopay adalah sebuah terobosan dalam dunia keua...,gopay adalah sebuah terobosan dalam dunia keua...,"[gopay, adalah, sebuah, terobosan, dalam, duni...","[terobos, dunia, uang, antarmuka, intuitif, fi...",terobos dunia uang antarmuka intuitif fiturfit...
14997,Saya beri nilai 1 karna apk ini tidak bisa di ...,Saya beri nilai karna apk ini tidak bisa di p...,saya beri nilai karna apk ini tidak bisa di p...,saya beri nilai karena aplikasi ini tidak bisa...,"[saya, beri, nilai, karena, aplikasi, ini, tid...","[nilai, tidak, percaya, modus, tipu, ajak, lin...",nilai tidak percaya modus tipu ajak link undan...
14998,Untuk saran ke developernya mungkin untuk pros...,Untuk saran ke developernya mungkin untuk pros...,untuk saran ke developernya mungkin untuk pros...,untuk saran ke developernya mungkin untuk pros...,"[untuk, saran, ke, developernya, mungkin, untu...","[saran, developer, proses, login, sandi, biome...",saran developer proses login sandi biometrik i...


In [10]:


# URL lexicon positif dan negatif
positive_url = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
negative_url = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"

# Membaca lexicon
positive_lexicon = set(pd.read_csv(positive_url, sep="\t", header=None)[0])
negative_lexicon = set(pd.read_csv(negative_url, sep="\t", header=None)[0])

# Fungsi untuk menentukan sentimen dan menghitung skornya
def determine_sentiment(text):
    if isinstance(text, str):
        positive_count = sum(1 for word in text.split() if word in positive_lexicon)
        negative_count = sum(1 for word in text.split() if word in negative_lexicon)
        sentiment_score = positive_count - negative_count

        if sentiment_score > 0:
            sentiment = "Positif"
        elif sentiment_score < 0:
            sentiment = "Negatif"
        else:
            sentiment = "Netral"

        return sentiment_score, sentiment
    return 0, "Netral"

df_clean[['score', 'sentimen']] = df_clean['text_ready'].apply(
    lambda x: pd.Series(determine_sentiment(x)))

# Lihat distribusi hasil
print("\nDistribusi sentimen hasil lexicon:")
print(df_clean['sentimen'].value_counts())
df_clean[['text_ready', 'score', 'sentimen']].head()


Distribusi sentimen hasil lexicon:
sentimen
Positif    6875
Negatif    4756
Netral     3350
Name: count, dtype: int64


/tmp/ipython-input-1529592151.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[['score', 'sentimen']] = df_clean['text_ready'].apply(
/tmp/ipython-input-1529592151.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[['score', 'sentimen']] = df_clean['text_ready'].apply(


,text_ready,score,sentimen
0,bagus gratis murah loh bintang pokok bug level...,1,Positif
1,tidak pindah mantap pokok cuma sayang sekali m...,-3,Negatif
2,bilang jaring putus pikir habis coba buka lanc...,5,Positif
3,koin² kumpul pet tidak akumulasi checkin misi ...,-3,Negatif
4,bagus transaksitetapi sekali erormasalah jarin...,0,Netral


In [11]:
# ==========SKEMA 1==========
# LINEAR SVC (SVM) | TF-IDF | 70:30


X = df_clean['text_ready']
y = df_clean['sentimen']

# TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_tfidf = vectorizer.fit_transform(X)

# SMOTE untuk balancing
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_tfidf, y)

# Split 70:30
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.3, random_state=42, stratify=y_res
)

# Optimasi parameter LinearSVC
param_grid = {
    'C': [0.1, 1, 10, 50, 100],
    'loss': ['hinge', 'squared_hinge'],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(LinearSVC(random_state=42, max_iter=5000),
                    param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best Parameters :", grid.best_params_)

# Prediksi
y_train_pred = grid.predict(X_train)
y_test_pred = grid.predict(X_test)

# Evaluasi akurasi
print(f"Akurasi Train: {accuracy_score(y_train, y_train_pred)*100:.2f}%")
print(f"Akurasi Test : {accuracy_score(y_test, y_test_pred)*100:.2f}%\n")

# Classification report
print("Classification Report (Test):\n", classification_report(y_test, y_test_pred))


Best Parameters : {'C': 1, 'class_weight': None, 'loss': 'squared_hinge'}
Akurasi Train: 98.86%
Akurasi Test : 89.29%

Classification Report (Test):
               precision    recall  f1-score   support

     Negatif       0.91      0.93      0.92      2062
      Netral       0.85      0.84      0.85      2063
     Positif       0.92      0.91      0.91      2063

    accuracy                           0.89      6188
   macro avg       0.89      0.89      0.89      6188
weighted avg       0.89      0.89      0.89      6188



In [12]:
# ==========SKEMA 2==========
# LSTM | 80:20

# Persiapan Data
X = df_clean["text_ready"].astype(str)
y = df_clean["sentimen"]

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Tokenisasi
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=100)

# Model LSTM
model1 = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=100),

    LSTM(128, dropout=0.4, recurrent_dropout=0.4, kernel_regularizer=l2(1e-4)),
    BatchNormalization(),

    Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.5),

    Dense(3, activation='softmax')
])

model1.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

# Callback (anti overfit)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, min_lr=1e-5)

# Training
history1 = model1.fit(
    X_train_seq, y_train,
    validation_data=(X_test_seq, y_test),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Evaluasi
train_loss1, train_acc1 = model1.evaluate(X_train_seq, y_train, verbose=0)
test_loss1, test_acc1 = model1.evaluate(X_test_seq, y_test, verbose=0)
print(f"\nAkurasi Training: {train_acc1*100:.2f}%")
print(f"Akurasi Testing : {test_acc1*100:.2f}%")


Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


188/188 ━━━━━━━━━━━━━━━━━━━━ 42s 202ms/step - accuracy: 0.5652 - loss: 0.9342 - val_accuracy: 0.6340 - val_loss: 0.9396 - learning_rate: 0.0010
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 42s 206ms/step - accuracy: 0.8217 - loss: 0.4605 - val_accuracy: 0.8682 - val_loss: 0.5386 - learning_rate: 0.0010
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 39s 209ms/step - accuracy: 0.8751 - loss: 0.3277 - val_accuracy: 0.8762 - val_loss: 0.3382 - learning_rate: 0.0010
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 41s 209ms/step - accuracy: 0.9162 - loss: 0.2423 - val_accuracy: 0.9066 - val_loss: 0.2619 - learning_rate: 0.0010
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 41s 211ms/step - accuracy: 0.9326 - loss: 0.1927 - val_accuracy: 0.9042 - val_loss: 0.2908 - learning_rate: 0.0010
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 40s 211ms/step - accuracy: 0.9536 - loss: 0.1451 - val_accuracy: 0.9112 - val_loss: 0.2874 - learning_rate: 0.0010
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 39s 209ms/step - accuracy: 0.9684 - loss:

In [13]:
# ==========SKEMA 3==========
# CNN | 80:20

X = df_clean['text_ready'].astype(str)
y = df_clean['sentimen']

# Encode label
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

# Tokenisasi dan padding
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)

# CNN Model
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    BatchNormalization(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Callback
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)

# Training
history = model.fit(
    X_train_seq, y_train,
    validation_data=(X_test_seq, y_test),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Evaluasi
y_train_pred = np.argmax(model.predict(X_train_seq), axis=1)
y_test_pred = np.argmax(model.predict(X_test_seq), axis=1)

from sklearn.metrics import accuracy_score, classification_report

print(f"\nAkurasi Train: {accuracy_score(y_train, y_train_pred)*100:.2f}%")
print(f"Akurasi Test : {accuracy_score(y_test, y_test_pred)*100:.2f}%\n")
print("Classification Report (Test Set):\n", classification_report(y_test, y_test_pred, target_names=le.classes_))


Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


188/188 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.5922 - loss: 0.8842 - val_accuracy: 0.7394 - val_loss: 0.8462 - learning_rate: 0.0010
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 10s 43ms/step - accuracy: 0.8680 - loss: 0.3273 - val_accuracy: 0.8478 - val_loss: 0.5312 - learning_rate: 0.0010
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - accuracy: 0.9290 - loss: 0.1938 - val_accuracy: 0.8659 - val_loss: 0.3494 - learning_rate: 0.0010
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 10s 42ms/step - accuracy: 0.9558 - loss: 0.1296 - val_accuracy: 0.8762 - val_loss: 0.3743 - learning_rate: 0.0010
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step - accuracy: 0.9684 - loss: 0.0945 - val_accuracy: 0.8719 - val_loss: 0.3856 - learning_rate: 0.0010
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 45ms/step - accuracy: 0.9850 - loss: 0.0526 - val_accuracy: 0.8775 - val_loss: 0.3809 - learning_rate: 5.0000e-04
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

Akur

In [20]:
import joblib

# Simpan model LinearSVC
joblib.dump(grid.best_estimator_, 'linear_svc_model.pkl')

# Simpan TF-IDF Vectorizer
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("berhasil disimpan!")


berhasil disimpan!


In [21]:

# Simpan model CNN
model.save('cnn_sentiment_model.keras')

# Simpan tokenizer
import pickle

with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("berhasil disimpan!")


berhasil disimpan!


In [24]:
# Simpan LSTM model
model1.save('lstm_sentiment_model.keras')

# Simpan tokenizer
with open('tokenizer_lstm.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("berhasil disimpan!")


from google.colab import files
files.download('lstm_sentiment_model.keras')
files.download('tokenizer_lstm.pkl')


berhasil disimpan!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>